# Preprocess JRA-3Q from glade storage
File list: https://rda.ucar.edu/datasets/d640000/dataaccess/#

In [1]:
import xarray as xr
import xesmf as xe
import numpy as np
import xagg as xa
import pandas as pd
import fsspec
from fsspec.implementations import local
from tqdm.notebook import tqdm
import re
import glob
import os
from numba import jit as njit
import zarr
import dask
from distributed import Client
import warnings

from funcs_support import get_filepaths, get_params, utility_save

dir_list = get_params()
# Get filesystem
fs = fsspec.implementations.local.LocalFileSystem()

In [ ]:
# Start dask client
client = Client()
display(client)

## Transfer and resample files, one-by-one

In [28]:
#data_loc = '/glade/campaign/collections/rda/data/d640000/anl_surf/'
#data_str = 'jra3q.anl_surf.0_0_0.tmp2m-hgt-an-gauss.'
#max_hr = '18'
#filevar = 'tmp2m-hgt-an-gauss'
#var = 'tas'

#data_loc = '/glade/campaign/collections/rda/data/d640000/minmax_surf/'
#data_str = 'jra3q.minmax_surf.0_0_0.tmpmax2m-hgt-fc-gauss.'
#max_hr = '23'
#filevar = 'tmpmax2m-hgt-fc-gauss'
#var = 'tasmax'

datesMS = pd.date_range('1981-01-01','2024-12-31',freq='1MS')
datesME = pd.date_range('1981-01-01','2024-12-31',freq='1ME')

fns = [(data_loc+re.sub(r'\-','',str(dateMS)[0:8])+'/'+
       data_str+
       re.sub(r'\-','',str(dateMS)[0:10])+'00_'+
       re.sub(r'\-','',str(dateME)[0:10])+max_hr+'.nc')
       for dateMS,dateME in zip(datesMS,datesME)]


operation = 'mean'
#operation = 'max'

In [29]:
for fn in tqdm(fns):
    # Get output filename
    fn_date = [comp for comp in re.split(r'\/',fn) if re.search('^[0-9]{6}$',comp)][0]
    fn_date = fn_date+'01-'+fn_date+str(pd.to_datetime(fn_date+'01').daysinmonth)
    output_fn = dir_list['raw']+'JRA-3Q/'+var+'_day_JRA-3Q_historical_reanalysis_'+fn_date+'.nc'
    
    if not fs.exists(output_fn):
        # Load data
        ds = xr.open_dataset(fn,chunks='auto')
        
        # Standardize
        ds = xa.fix_ds(ds)
        
        # Rename / remove unneeded variables
        ds = ds.rename({filevar:var})
        ds = ds[[var]]
        
        # Resample to daily
        if operation == 'mean':
            ds = ds.resample({'time':'1D'}).mean()
        elif operation == 'max':
            ds = ds.resample({'time':'1D'}).max()
        
        ds.attrs['SOURCE'] = 'preprocess_JRA3Q.ipynb'
        ds.attrs['DESCRIPTION'] = 'JRA-3Q model grid data, standardized and resampled from 6-hourly to daily'
    
        utility_save(ds,output_fn)
    else:
        print(output_fn+' already exists, skipped!')

  0%|          | 0/288 [00:00<?, ?it/s]

/glade/derecho/scratch/schwarzwald/climate_raw_derecho/JRA-3Q/pr_day_JRA-3Q_historical_reanalysis_19570101-19570131.nc saved!
/glade/derecho/scratch/schwarzwald/climate_raw_derecho/JRA-3Q/pr_day_JRA-3Q_historical_reanalysis_19570201-19570228.nc saved!
/glade/derecho/scratch/schwarzwald/climate_raw_derecho/JRA-3Q/pr_day_JRA-3Q_historical_reanalysis_19570301-19570331.nc saved!
/glade/derecho/scratch/schwarzwald/climate_raw_derecho/JRA-3Q/pr_day_JRA-3Q_historical_reanalysis_19570401-19570430.nc saved!
/glade/derecho/scratch/schwarzwald/climate_raw_derecho/JRA-3Q/pr_day_JRA-3Q_historical_reanalysis_19570501-19570531.nc saved!
/glade/derecho/scratch/schwarzwald/climate_raw_derecho/JRA-3Q/pr_day_JRA-3Q_historical_reanalysis_19570601-19570630.nc saved!
/glade/derecho/scratch/schwarzwald/climate_raw_derecho/JRA-3Q/pr_day_JRA-3Q_historical_reanalysis_19570701-19570731.nc saved!
/glade/derecho/scratch/schwarzwald/climate_raw_derecho/JRA-3Q/pr_day_JRA-3Q_historical_reanalysis_19570801-19570831.nc

## Combine to single file

In [30]:
fns = glob.glob(dir_list['raw']+'JRA-3Q/'+var+'_day_*_*-19*.nc')

In [31]:
dss = xr.open_mfdataset(fns).chunk({'time':30})

In [32]:
timestr = (re.sub(r'\-','',str(dss.time.min().values)[0:10])+'-'+
           re.sub(r'\-','',str(dss.time.max().values)[0:10]))
timestr

'19570101-19801231'

In [33]:
utility_save(dss,dir_list['raw']+'JRA-3Q/'+var+'_day_JRA-3Q_historical_reanalysis_'+timestr+'.zarr')

/glade/u/home/schwarzwald/.conda/envs/hle_iv/lib/python3.13/site-packages/zarr/api/asynchronous.py:227: UserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


/glade/derecho/scratch/schwarzwald/climate_raw_derecho/JRA-3Q/pr_day_JRA-3Q_historical_reanalysis_19570101-19801231.zarr saved!


In [34]:
for fn in np.sort(fns):
    os.remove(fn)
    print('.',end='')

................................................................................................................................................................................................................................................................................................